---
downloads:
  - id: lecture-script
    title: Lecture Script (.pdf)
  - file: session5_demos.ipynb
    title: Jupyter Notebook (.ipynb)
  - file: ../pyproject.toml
    title: PyProject Config (.toml)
  - file: ../downloads/streamlit_example.py
    title: Streamlit Example (.py)
---

# Session 5: Dashboards and Demos

⚠️ <span style="color: orange;">Please note we are currently reworking the lecture and still preparing the course materials. You will find this message wherever the course materials are being updated / not finalized.</span> ⚠️

## Learning goals

- Build interactive dashboards to explore ML outputs
- Communicate findings effectively to non-technical stakeholders

## Streamlit
- [Streamlit](https://streamlit.io) is an open-source Python library that makes it easy to create and share web apps
- Easy-to-read code
- Popular in the data science community for building interactive dashboards
- As simple as creating a plot or writing a print statement

### Why Streamlit in this course

- Building dashboards is important for communicating results to stakeholders
- Fast path from modeling results to shareable interface
- Good for model comparison, what-if interaction, and error analysis views

- To get started you can install Streamlit using pip:
    ```bash
    pip install streamlit
    streamlit hello
    ```

### Example Streamlit app
- We have a simple Streamlit app in [`streamlit_example.py`](../downloads/streamlit_example.py) that simulates a spam detection workflow:

```python
import numpy as np
import streamlit as st

def random_email():
	return {
		"subject": str(np.random.choice(["urgent invoice", "meeting reminder", "free gift", "account update"])),
		"body": str(np.random.choice(["please review", "click now", "open attachment", "see details"])),
	}

def model_predict(email, spam_probability):
    np.random.seed(hash(email.__str__()) % (2**8)) # Seed based on email content
    return np.random.choice(["Spam", "Not spam"], p=[spam_probability, 1 - spam_probability])

st.set_page_config(page_title="DSW Spam Demo", page_icon="📧")
st.title("DSW Spam Detection Demo")
st.write("A tiny example for showing a spam classifier workflow. Uses deterministic random predictions to simulate a model.")

if "email" not in st.session_state:
	st.session_state.email = random_email()

if st.button("Generate random email"):
	st.session_state.email = random_email()
	st.rerun()

st.subheader("Message")
title = st.text_input("Title", value=st.session_state.email["subject"])
body = st.text_area("Body", value=st.session_state.email["body"])
spam_probability = st.slider("Spam probability", 0.0, 1.0, 0.5, 0.05)

if st.button("Predict"):
    st.session_state.email = {"subject": title, "body": body}
    prediction = model_predict(st.session_state.email, spam_probability)
    
    if prediction == "Spam":
        st.error(f"Prediction: {prediction}")
    else:
        st.success(f"Prediction: {prediction}")
```

To run locally, download the file (and dependencies) and run:
```bash
streamlit run streamlit_example.py
```

---

**📸 Screenshot:** _(Insert a screenshot of the running Streamlit app here)_

### How to host Streamlit apps

**Streamlit Community Cloud**:
- The easiest (and free)option - connect your GitHub repo and it automatically deploys your app
- Visit [streamlit.io/cloud](https://streamlit.io/cloud)
- Best for public apps and prototypes

**Self-hosted options**:
- **Docker** containerize your app and deploy the container
- You can use our [Kubernetes cluster](https://docs.cluster.ris.bht-berlin.de) (see [session 2](DSW-sessions/session2_cluster.ipynb)) to deploy

**Basic deployment steps**:
```bash
# Install dependencies
pip install -r requirements.txt

# Run locally to test
streamlit run your_app.py
```

### Further Streamlit examples
- Visit the [Streamlit App-Gallery](https://streamlit.io/gallery) for more examples of what you can build with Streamlit
- They also let you fork and deploy

## Backend with FastAPI

So far we've built interactive UIs with Streamlit. For simple demos, Streamlit runs **standalone** — the UI and model logic live in the same process. This is perfectly fine for prototyping.

But in production ML systems, you often want to **separate the backend from the frontend**. This allows you to:

- Scale the model serving independently from the UI
- Serve the same model from multiple clients (web app, mobile app, etc.)
- Keep your model infrastructure decoupled and testable

This is where **FastAPI** comes in — it's a modern Python web framework for building robust APIs that can host your model.

### Minimal example

```python
from fastapi import FastAPI
import joblib

app = FastAPI()
model = joblib.load('model.pkl')

@app.get('/predict')
def predict(features: dict):
    prediction = model.predict([features['input']])
    return {'prediction': prediction.tolist()}
```

### Running the API

```bash
uvicorn main:app --reload
```

### Testing the API

- Visit `http://localhost:8000/docs` for interactive Swagger UI
- Or use `curl`:

```bash
curl -X POST 'http://localhost:8000/predict' \
  -H 'Content-Type: application/json' \
  -d '{"input": [1.0, 2.0, 3.0]}'
```

## Some Demo Showcases

- [SoilNet app](https://soilnet.demo.calgo-lab.de): Interactive segmentation and classification of soil images
- [RIWWER dashboard](https://riwwer.demo.calgo-lab.de): Shows a simulated live view of time series forecasts for sewer system management

## Summary & Key Takeaways

| Aspect | Key Tool | Key Finding |
|---|---|---|
| **Interactive UI** | Streamlit | Fast path from ML results to shareable web app |
| **Prototyping** | `streamlit run` | Standalone app, no separate backend needed for demos |
| **Deployment** | Streamlit Community Cloud | Free hosting via GitHub connect |
| **Production scale** | FastAPI + uvicorn | Decouple frontend from model serving |
| **Real-world examples** | SoilNet, RIWWER, Streamlit App Gallery | Dashboards and demos to get some ideas |

> **Bottom line**: Start with Streamlit for prototypes. Scale to FastAPI when you need multiple clients or independent model serving.